# 전이학습

### 딥러닝 모델의 한계

- 흉부 X-ray 100장으로 정상·폐렴 분류 모델을 처음부터 만들면 데이터가 부족해 패턴을 다 못 배우고 과적합이 생기기 쉬우며 학습 시간과 비용도 많이 든다.

=> 이미 많은 이미지를 학습한 모델의 특징 파악 능력을 가져와 쓰자는 게 전이학습이다.

### 전이학습 정의 및 분류

1. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**전이학습 정의**</mark> :
    사전 학습으로 배운 능력을 활용해 새로운 작업을 학습하는 것이다. 동물 사진으로 학습한 모델의 특징 추출 능력은 그대로 쓰고, 분류층만 정상, 폐렴용으로 바꿔 X-ray로 학습한다.

    | 용어 | 뜻 | 예시 |
    | --- | --- | --- |
    | Source | 기존에 학습한 문제 | 고양이·강아지 분류 |
    | Target | 새로 해결할 문제 | 정상·폐렴 분류 |
    | Domain | 데이터의 특성 | 동물 사진 / X-ray |
    | Task | 수행하려는 작업 | 동물 분류 / 질환 분류 |

    ***전이학습 조건** : Source와 Target의 Domain이나 Task 중 하나라도 다를 때이다.

2. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Domain과 Task에 따른 분류**</mark> :
    - 귀납적 : Task가 다르고 Target 정답 레이블이 있다. (이미지 분류 → 객체 탐지)
    - 변환적 : Domain만 다르고 Target 레이블이 없다. (뉴스 감정 분석 → 영화 리뷰 감정 분석)
    - 비지도 : 둘 다 레이블이 없다. (한 분야 군집화 → 다른 분야 군집화)

3. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**전이 대상에 따른 분류**</mark> :
    가져오는 대상에 따라 인스턴스, 특성, 파라미터, 관계 기반으로 나뉜다. 지금은 주로 파라미터 기반을 쓴다.

### 전이학습 패러다임 : Pre-training & Fine-tuning

- 전통적인 방식은 라벨이 붙은 ImageNet으로 CNN을 학습해 가중치를 재사용하거나 미리 만든 단어 벡터를 재사용했다. 단어 벡터는 '배를 먹었다'와 '배를 탔다'의 '배'를 구분하지 못한다는 한계가 있었다.

- 트랜스포머 이후에는 이미지 일부를 가리고 복원하는 자기지도학습으로 라벨 없이도 특징을 배우고, BERT나 GPT처럼 문맥을 반영한 표현을 배운다.

1. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Foundation Model**</mark> :
    여러 작업의 기반이 되도록 방대한 데이터로 사전 학습한 모델이다. 하나의 언어 모델을 상담 데이터로 학습하면 상담 모델, 회의록으로 학습하면 요약 모델이 된다.

### Fine-tuning & PEFT

1. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Fine-tuning**</mark> :
    사전 학습된 모델을 새 데이터로 추가 학습해 가중치 일부 또는 전체를 조정하는 것이다.

    - Full : 모든 가중치를 학습한다. 많이 바꿀 수 있지만 비용이 크고 기존 지식을 잊을 수 있다.
    - Partial : 선, 윤곽 같은 일반 특징은 고정하고 뒤쪽 층만 학습한다.
    - Frozen Backbone : 새 분류층만 학습한다. 가장 가볍지만 적응에 한계가 있다.

    ***선택 기준** : 데이터가 적고 도메인이 비슷하면 Frozen Backbone, 데이터가 많고 도메인이 다르면 Full, 그 사이면 Partial을 쓴다.

2. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**PEFT**</mark> :
    가중치 대부분을 고정하고 최소한의 파라미터만 학습하는 방법이다. 기본 모델은 한 번만 저장하고 작업별 작은 파라미터만 따로 저장하면 돼서 메모리 부담이 적다.

    ***LoRA** : 기존 가중치 $W$는 고정하고, 수정분을 작은 두 행렬의 곱 $BA$로 학습한다. 1000 × 1000 층이면 원래 100만 개를 학습해야 하지만, r = 8이면 16,000개만 학습하면 된다. r이 클수록 표현력은 좋아지지만 비용도 늘어난다.

    ***Adapter / Prompt tuning** : 모델 중간에 작은 신경망을 끼우거나, 입력 앞에 학습 가능한 벡터를 붙여 그 부분만 학습한다.

### 전이학습의 한계와 활용 사례

1. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**한계**</mark> :
    - Negative Transfer : 전이 후 오히려 성능이 떨어진다. 분포 차이는 MMD로 확인한다.
    - Catastrophic Forgetting : 새로 배우면서 기존 능력을 잊는다. 중요한 가중치 변화에 패널티를 주는 EWC로 줄인다.
    - Bias Transfer : 사전 학습의 편향이 그대로 이어진다.
    - Transferability 측정의 어려움 : 어떤 모델이 잘 맞을지 미리 알기 어렵다. 기존 모델의 예측 확률을 쓰는 LEEP, 특징 벡터를 쓰는 LogME로 Fine-tuning 전에 추정한다.

2. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**활용 사례**</mark> :
    일반 이미지 모델을 X-ray로 Fine-tuning해 폐렴을 분류하거나 언어 모델을 계약서로 Fine-tuning해 조항 유형을 분류한다.

### 딥러닝 모델 경량화

- 학습이 끝난 모델을 실제로 쓸 때 드는 메모리와 연산을 줄이는 방법이다.

| 방법 | 내용 |
| --- | --- |
| 가지치기 | 영향이 작은 가중치나 계산 묶음을 제거 |
| 양자화 | 숫자를 더 적은 비트로 표현 (32비트 → 8비트면 1/4) |
| 지식 증류 | 작은 Student가 큰 Teacher의 출력 분포를 따라 학습 |
| 구조 경량화 | 처음부터 계산이 적은 구조로 설계 |

=> 재학습이 어려우면 양자화, 데이터와 자원이 있으면 가지치기나 지식 증류를 고려한다. 대신 줄일수록 성능이 떨어질 수 있다.

# AI Alignment

### AI 정렬이 필요한 이유와 정렬 기준

- LLM은 자연스러운 문장은 잘 만들지만 사실과 다른 말을 하거나 편향을 드러낼 수 있다. 그래서 바람직한 답의 기준을 정하고 모델을 그에 맞게 조정하는 AI Alignment가 필요하다.

- 정렬 기준은 도움이 되는지, 사실에 근거하는지, 해가 없는지이다.

- 다만 사람마다 가치관이 다르고 모든 답을 사람이 검토할 수 없어서 정렬은 어렵다.

### AI 정렬의 두 가지 관점

1. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Outer Alignment**</mark> :
    목표를 제대로 정했는지를 본다. 대표적인 문제가 보상 해킹으로, "방 치워줘"에 보이는 물건이 적을수록 점수를 주면 로봇이 정리 대신 카메라를 가려버리는 식이다.

2. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Inner Alignment**</mark> :
    모델이 실제로 무엇을 배웠는지를 본다. 감독받을 때만 의도대로 행동할 수도 있다는 기만적 정렬 가능성이 있어서, 모델 내부를 들여다보는 해석 가능성 연구가 중요하다.

### RLHF

- 사람의 피드백으로 사람이 선호하는 답을 하도록 모델을 조정하는 방법이다.

1. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**파이프라인**</mark> : SFT → Reward Model → RL
    - SFT : 사람이 쓴 모범 답변으로 학습한다.
    - Reward Model : 여러 답에 사람이 순위를 매기고, 선호한 답에 높은 점수를 주도록 학습한다.
    - RL : 점수가 높은 답을 더 자주 내도록 조정한다. PPO와 KL 패널티로 너무 급하게 바뀌지 않게 막는다.

2. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**부작용**</mark> :
    점수만 높이려는 Reward Hacking, 사실보다 사용자 말에 무조건 동의하는 Sycophancy가 생길 수 있다.

# 프롬프트 엔지니어링

### Prompt vs Fine-tuning

- 모델은 그대로 두고 더 좋은 답이 나오도록 입력을 설계하는 기술이다. "서울 맛집 알려줘"보다 "홍대 근처 1인 2만 원 이하 혼밥 한식집 3곳, 표로"처럼 조건을 주면 바로 쓸 수 있는 답이 나온다.

| 구분 | 프롬프트 | Fine-tuning |
| --- | --- | --- |
| 비용 | 낮음 | 높음 |
| 속도 | 바로 수정 가능 | 학습 시간 필요 |
| 한계 | 모델이 원래 가진 능력 안에서만 | 모델 자체를 도메인에 맞춤 |

=> 보통 프롬프트부터 해보고, 안 되면 Fine-tuning을 고려한다.

### 주요 기법

1. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Zero-shot / Few-shot**</mark> :
    예시 없이 지시만 주면 Zero-shot, 예시를 몇 개 보여주면 Few-shot이다. 리뷰 감정 분류에서 "장단점이 섞이면 중립" 같은 예시를 넣어주면 판단 기준이 맞춰진다.

2. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Chain-of-Thought (CoT)**</mark> :
    중간 풀이를 단계별로 쓰게 하는 방식이다. "손흥민이 월드시리즈 투수로 나섰다"를 바로 판단하지 말고, 사람과 대회를 먼저 확인하게 하면 덜 틀린다.

3. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Role Prompting**</mark> :
    역할을 줘서 관점과 말투를 바꾼다. 다만 역할을 줘도 모델이 아는 내용이 늘어나진 않는다.

4. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Instruction Prompting**</mark> :
    형식, 길이, 대상, 제약을 구체적으로 지시한다. 꼭 들어가야 할 내용도 적어줘야 빠지지 않는다.

5. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**Self-Consistency**</mark> :
    여러 번 풀게 하고 다수결로 답을 고른다. 정확해지지만 비용이 풀이 횟수만큼 늘어난다.

=> 보통 Instruction으로 시작해서, 형식이 필요하면 Few-shot, 여러 단계면 CoT, 정확도가 중요하면 Self-Consistency를 더한다.

### 프롬프트 엔지니어링의 한계

- Prompt Injection : 시스템 프롬프트와 사용자 입력이 한 덩어리로 들어가서, 입력에 "위 지시는 무시해" 같은 문장을 넣으면 모델이 따를 수 있다. 구분자, 권한 최소화, 출력 검증으로 완화한다.

- Hallucination : 없는 논문이나 숫자를 그럴듯하게 지어낸다. 근거 문서를 넣어주는 RAG나 출처 확인으로 완화한다.

- 모델 의존성 : 모델 버전이 바뀌면 같은 프롬프트도 결과가 달라진다. 버전을 고정하고 회귀 테스트를 한다.

- 성능 한계 : 큰 수 계산처럼 원래 못하는 일은 프롬프트로 해결 안 된다. 외부 도구를 붙이거나 Fine-tuning한다.

=> 도구 연결, 권한, 검증은 프롬프트로 못 하고 모델 바깥의 시스템을 바꿔야 한다. 그게 하네스이다.

# 하네스 엔지니어링

### 하네스의 정의와 구성 요소

- 도구를 쓰며 여러 단계를 스스로 진행하는 AI를 에이전트라고 하고, 에이전트는 모델 본체와 그걸 둘러싼 시스템인 **하네스**로 나뉜다.

- 모델은 "이 파일을 읽겠다", "12번째 줄을 고치자"처럼 할 일을 말할 뿐이고, 실제로 파일을 열고 고치고 테스트하는 건 하네스이다. 요리사가 일하려면 주방이 필요한 것과 같다.

| 요소 | 고객센터 에이전트 예시 |
| --- | --- |
| 지시/컨텍스트 | 시스템 프롬프트, 고객 정보, 이전 대화 |
| 도구 인터페이스 | 주문 조회 도구 정의 |
| 실행 환경 | 주문 DB 서버, 샌드박스 |
| 상태/메모리 | 조회한 결과, 진행 단계 |
| 관측/피드백 | 도구 반환값, 오류 메시지 |
| 권한/가드레일 | 조회만 허용, 환불은 사람 승인 |

***도구 스키마** : 모델은 도구 코드를 안 보고 스키마만 보고 쓴다. 설명이 모호하면 엉뚱한 값을 넣기 때문에 언제 쓰는지, 어떤 값을 넣는지 명확히 적어야 한다.

***반환 계약** : 주문이 없는 경우와 DB 오류로 실패한 경우를 둘 다 `None`으로 돌려주면 모델이 구분할 수 없다. 성공 여부, 오류 코드, 재시도 가능 여부를 나눠서 돌려줘야 다음 행동을 정할 수 있다.

### 에이전트 실패 유형 별 하네스 개선 방법

- 에이전트가 실패하면 모델 성능 뿐만 아니라 인터페이스를 봐야 한다. SWE-agent 연구에서는 모델은 그대로 두고 ACI만 바꿨는데 버그 해결률이 11.0%에서 18.0%로 올랐다.

| 증상 | 고칠 곳 |
| --- | --- |
| 엉뚱한 도구를 부른다 | 도구 이름, 설명, 입력 형식 |
| 같은 실수를 반복한다 | 반환값, 오류 메시지 |
| 앞부분을 잊는다 | 넘기는 정보, 진행 기록, 요약 |
| 위험한 행동을 바로 한다 | 권한, 사람 승인, 샌드박스 |
| 결과가 맞는지 모른다 | 테스트, 완료 조건 |

# 루프 엔지니어링

### 루프의 정의와 ReAct

- 사람이 직접 하던 "답 확인 → 다시 시키기"를 시스템이 반복하게 만든 게 루프이다. 한 바퀴는 관측 → 판단 → 행동 → 검증이고, 검증 결과가 다음 바퀴의 관측이 된다.

- 하네스가 일하는 공간을 설계한다면, 루프는 시간에 따른 반복을 설계한다.

1. ##### <mark style="background-color: rgba(135, 206, 235, 0.7);">**ReAct**</mark> :
    생각 → 행동 → 관측을 번갈아 하는 방식으로, 루프의 뿌리가 된 연구이다. 생각만 하면 지어내고 행동만 하면 같은 행동을 반복하는데 둘을 번갈아 하면서 이런 문제를 줄였다.

### 루프의 구성 요소

- 루프를 만들려면 목표, 트리거, 관측과 행동, 정상 종료 조건, 안전 상한, 실패 승인 처리를 정해야 한다.

- 멈추는 조건은 두 가지이다. 목표를 달성해서 멈추는 정상 종료 조건과, 반복 횟수·시간·비용이 넘어서 멈추는 안전 상한이다. 정상 종료 조건이 영원히 참이 안 될 수도 있기 때문에 안전 상한이 꼭 있어야 한다.

- 모델이 끝났다고 해도 실제로 끝났는지는 테스트 같은 외부 기준으로 확인해야 한다.

### 통제 장치와 실행 단계

- 반복하면 잘못된 행동도 같이 반복된다. 무한 반복, 비용 폭주, 같은 메일 여러 번 발송, 확인 없는 삭제 같은 문제가 커질 수 있다.

- 그래서 반복, 시간, 비용 상한, 테스트, 중복 실행 방지, 체크포인트와 롤백을 둔다. 행동도 위험도에 따라 나눠서 조회는 바로 실행하고 파일 수정은 실행 후 확인하고 삭제나 결제는 사람이 승인해야 실행한다.

- RLHF가 모델 안을 바꾸는 학습 단계의 정렬이라면 이건 실행 단계의 통제이다. 모델이 Prompt Injection에 속아 환불을 하려 해도 사람 승인이 있어야 실행되게 해두면 실제 피해는 생기지 않는다.

=> 프롬프트는 입력을, 하네스는 환경을, 루프는 반복을 바꾼다. 셋 다 모델은 그대로 두고 결과를 바꾸는 방법이다.